# 13 · Training runner: EdNet-KT1 and English Learning Events

Supports thesis Section 4.3. Runs `knowledge_tracing/run_benchmark.sh` for one dataset: prepare → (optional validation-only search) → train DKT, SAKT, AKT, DKVMN, SAINT × seeds 42/43/44 → evaluate on the test fold → hardening (BKT, logistic regression, bootstrap CIs, paired differences). Everything is resumable: a finished (model, seed) with the same configuration is reused, never retrained.

**Local (recommended):** run from the repository with `.venv-kt` on the GPU.
**Colab:** set `ON_COLAB_DRIVE = True`; it mounts `MyDrive/ednet_kt1`, unpacks `ednet_colab_bundle.zip`, links `data/` to Drive (outputs survive a disconnect) and installs pyKT. If a session runs out of GPU time, run all cells again: only the interrupted run restarts.

In [1]:
import sys, os
if "google.colab" in sys.modules:
    !pip install -q seaborn
from pathlib import Path
_root = Path.cwd().resolve()
while not (_root / "knowledge_tracing").is_dir() and _root.parent != _root:
    _root = _root.parent
if "google.colab" in sys.modules and not (_root / "knowledge_tracing").is_dir():
    import zipfile
    from google.colab import drive
    drive.mount("/content/drive")
    import time as _t
    with zipfile.ZipFile("/content/drive/MyDrive/ednet_kt1/ednet_colab_bundle.zip") as zf:
        for _i in zf.infolist():   # keep the zip's timestamps: the runner compares mtimes
            _p = zf.extract(_i, "/content/ednet")
            _m = _t.mktime(_i.date_time + (0, 0, -1))
            os.utime(_p, (_m, _m))
    _root = Path("/content/ednet")
sys.path[:0] = [str(_root), str(_root / "knowledge_tracing")]
import report_utils as ru
ROOT = ru.setup()
ru.style()
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
pd.set_option("display.width", 160, "display.max_columns", 30)
print("repository:", ROOT)

repository: /home/rene/MicroLearningThesis


## 1. Settings

In [2]:
DATASET = "ednet_kt1"          # "ednet_kt1" | "english_bench"
SIZES = [300, 1000, 2000, 5000, 10000, 20000]   # EdNet cohorts (ignored for english_bench)
SEEDS = [42, 43, 44]
TRAIN_MODELS = ["dkvmn", "saint"]               # models to train; finished runs are reused anyway
EVAL_MODELS = ["dkt", "sakt", "akt", "dkvmn", "saint"]
TUNE_MODELS = []                                # e.g. ["dkvmn", "saint"] for english_bench
EVAL_FROM_DUMPS = ["dkt", "sakt", "akt"] if DATASET == "ednet_kt1" else []   # EdNet run-1 checkpoints were not kept: score their saved predictions
EPOCHS = None                                   # None = configured 50 (early stop after 10 flat epochs)
DRY_RUN = True                                  # True: show status only; set False to train

## 2. Environment

In [3]:
import subprocess, hashlib, time
PY = sys.executable if ru.in_colab() else str(ROOT / ".venv-kt/bin/python")
if ru.in_colab():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "pykt-toolkit==0.0.38"], check=True)
out = subprocess.run([PY, "-c", "import torch, numpy; from importlib.metadata import version; print('torch', torch.__version__, '| GPU', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none', '| numpy', numpy.__version__, '| pykt-toolkit', version('pykt-toolkit'))"],
                     capture_output=True, text=True)
print(out.stdout or out.stderr)

torch 2.14.1+cu126 | GPU NVIDIA GeForce GTX 1070 Ti | numpy 2.4.6 | pykt-toolkit 0.0.38



## 3. Status and protection
Every file of a finished run (summary and checkpoint) is fingerprinted before the run; afterwards the runner asserts that none of them changed.

In [4]:
BENCH = Path("data/benchmarks")
def workspaces():
    if DATASET == "english_bench":
        return {"english": BENCH / "english_kt/kt"}
    return {n: BENCH / f"ednet_kt1_{n}/kt" for n in SIZES}

def status():
    rows = []
    for key, kt in workspaces().items():
        for m in EVAL_MODELS:
            done = [s for s in SEEDS if (kt / "artifacts" / m / f"seed{s}" / "run_summary.json").exists()]
            rows.append({"workspace": key, "model": m.upper(), "finished seeds": len(done)})
    return pd.DataFrame(rows).pivot(index="workspace", columns="model", values="finished seeds")

def fingerprint():
    fp = {}
    for kt in workspaces().values():
        for f in (kt / "artifacts").glob("*/seed*/*"):
            if f.name in ("run_summary.json", "qid_model.ckpt") and (f.parent / "run_summary.json").exists():
                fp[str(f)] = hashlib.sha256(f.read_bytes()).hexdigest()
    return fp

display(status())

model,AKT,DKT,DKVMN,SAINT,SAKT
workspace,,,,,
300,3,3,3,3,3
1000,3,3,3,3,3
2000,3,3,3,3,3
5000,3,3,3,3,3
10000,3,3,3,3,3
20000,3,3,3,3,3


## 4. Run

In [5]:
if DRY_RUN:
    print("DRY_RUN is True: nothing trained.")
else:
    before = fingerprint()
    env = dict(os.environ, PY=PY, DATASET=DATASET, SIZES=" ".join(map(str, SIZES)),
               SEEDS=" ".join(map(str, SEEDS)), TRAIN_MODELS=" ".join(TRAIN_MODELS),
               EVAL_MODELS=" ".join(EVAL_MODELS), TUNE_MODELS=" ".join(TUNE_MODELS),
               EVAL_FROM_DUMPS=" ".join(EVAL_FROM_DUMPS))
    if EPOCHS:
        env["EPOCHS"] = str(EPOCHS)
    log = BENCH / f"run_{DATASET}_{time.strftime('%Y%m%d_%H%M%S')}.log"
    t0 = time.time()
    with open(log, "w") as lf:
        p = subprocess.Popen(["bash", "knowledge_tracing/run_benchmark.sh"], env=env,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            lf.write(line)
            if line.startswith(("===", "---")):
                print(line, end="")
        rc = p.wait()
    print(f"exit {rc}, {(time.time() - t0) / 3600:.2f} h, log: {log}")
    after = fingerprint()
    changed = [f for f in before if after.get(f) != before[f]]
    assert not changed, f"finished runs were modified: {changed[:5]}"
    print(f"OK: {len(before)} files of finished runs unchanged; {len(after) - len(before)} new files")
    display(status())

DRY_RUN is True: nothing trained.
